# 10 · Changing data: INSERT, UPDATE, MERGE, SCD Type 2, incremental loads (BigQuery)

**DML** (Data Manipulation Language) changes rows: `INSERT`, `UPDATE`, `DELETE`, `MERGE`.
**DDL** (Data Definition Language) changes tables: `CREATE`, `ALTER`, `DROP`.

This notebook works on **copies** named `prac_*`, so the 7 practice tables never change.
Each question has a setup cell that rebuilds its copy: run it first, and again to start over.

Data engineering words used here:
- **upsert** = update the row if the key exists, insert it if not (`MERGE`)
- **SCD Type 2** (slowly changing dimension) = keep the **history**: close the old row, add a new current row
- **incremental load** = load only the new rows since last time (using a **watermark**, e.g. the max date already loaded)
- **idempotent** = running the job twice gives the same result as running it once

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `shield.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`bigquery/practice/10_dml_merge_scd.ipynb`](../../bigquery/practice/10_dml_merge_scd.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. INSERT a row  ·  🟢 easy

Add customer **41**, `Test User`, city `Austin`, signup `2024-12-01` to `prac_customers`. Then show `num_customers` (the row count).

*Expected result: 1 row · columns: `num_customers`*

**How it works:** `INSERT INTO table (columns) VALUES (...)` adds one row; the SELECT COUNT checks it is there.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_customers AS
SELECT * FROM shield.customers

OK (create table as select)

In [ ]:
%%bigquery
INSERT INTO shield.prac_customers (customer_id, name, city, signup_date)
VALUES (41, 'Test User', 'Austin', DATE '2024-12-01')

OK, 1 row(s) affected

In [ ]:
%%bigquery
SELECT COUNT(*) AS num_customers FROM shield.prac_customers

,num_customers
0,41


**Why / traps:** Always list the columns in INSERT. If someone adds a column later, your insert still works.

## Q2. UPDATE with a condition  ·  🟢 easy

In `prac_customers`, set `city` to `Unknown` where it is NULL. Then show `customer_id`, `name`, `city` of the changed rows.

*Expected result: 1 row · columns: `customer_id`, `name`, `city`*

**How it works:** `UPDATE ... SET city = 'Unknown' WHERE city IS NULL` changes only the matching rows.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_customers AS
SELECT * FROM shield.customers

OK (create table as select)

In [ ]:
%%bigquery
UPDATE shield.prac_customers SET city = 'Unknown' WHERE city IS NULL

OK, 1 row(s) affected

In [ ]:
%%bigquery
SELECT customer_id, name, city FROM shield.prac_customers WHERE city = 'Unknown'

,customer_id,name,city
0,13,Lucas Lopez,Unknown


**Why / traps:** Write the WHERE first and test it as a SELECT. An UPDATE without WHERE changes **every** row.

BigQuery requires a WHERE in UPDATE and DELETE. To change all rows write `WHERE TRUE`.

## Q3. DELETE with NOT EXISTS  ·  🟢 easy

Delete from `prac_customers` everyone who **never ordered**. Then show `num_customers`.

*Expected result: 1 row · columns: `num_customers`*

**How it works:** `DELETE ... WHERE NOT EXISTS (order for this customer)` removes customers who never ordered.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_customers AS
SELECT * FROM shield.customers

OK (create table as select)

In [ ]:
%%bigquery
DELETE FROM shield.prac_customers c
WHERE NOT EXISTS (SELECT 1 FROM shield.orders o WHERE o.customer_id = c.customer_id)

OK, 8 row(s) affected

In [ ]:
%%bigquery
SELECT COUNT(*) AS num_customers FROM shield.prac_customers

,num_customers
0,32


**Why / traps:** `TRUNCATE TABLE` removes **all** rows fast (no WHERE). In Oracle it cannot be rolled back.

## Q4. Upsert with MERGE  ·  🟡 medium

`prac_customer_updates` has 4 rows: customers 1 and 2 changed, 41 and 42 are new. MERGE them into `prac_customers`: update `name` and `city` when the id exists, insert when not. Then show customers 1, 2, 41, 42.

*Expected result: 4 rows · columns: `customer_id`, `name`, `city`, `signup_date`*

**How it works:** MERGE matches source to target on the key: `WHEN MATCHED` → UPDATE the row, `WHEN NOT MATCHED` → INSERT it. One statement does both.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_customers AS
SELECT * FROM shield.customers

OK (create table as select)

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_customer_updates (
    customer_id INT64, name STRING, city STRING, signup_date DATE)

OK (create table)

In [ ]:
%%bigquery
INSERT INTO shield.prac_customer_updates VALUES
    (1,  'Aarav Nguyen',    'Denver',  DATE '2023-01-26'),
    (2,  'Priya Silva-Rao', 'Seattle', DATE '2023-09-08'),
    (41, 'Nina Patel',      'Boston',  DATE '2024-12-05'),
    (42, 'Omar Haddad',     'Austin',  DATE '2024-12-06')

OK, 4 row(s) affected

In [ ]:
%%bigquery
MERGE shield.prac_customers t
USING shield.prac_customer_updates s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN
  UPDATE SET name = s.name, city = s.city
WHEN NOT MATCHED THEN
  INSERT (customer_id, name, city, signup_date)
  VALUES (s.customer_id, s.name, s.city, s.signup_date)

OK, 4 row(s) affected

In [ ]:
%%bigquery
SELECT * FROM shield.prac_customers
WHERE customer_id IN (1, 2, 41, 42)
ORDER BY customer_id

,customer_id,name,city,signup_date
0,1,Aarav Nguyen,Denver,2023-01-26
1,2,Priya Silva-Rao,Seattle,2023-09-08
2,41,Nina Patel,Boston,2024-12-05
3,42,Omar Haddad,Austin,2024-12-06


**Why / traps:** MERGE fails (or does random things) if the **source has 2 rows for one key**. Dedupe the source first. MERGE is the main tool for upserts in data warehouses.

BigQuery MERGE can also `WHEN NOT MATCHED BY SOURCE THEN DELETE` (delete target rows that are missing in the source).

## Q5. Idempotent reload of one slice  ·  🟡 medium

`prac_orders` is a raw copy of `orders` (with duplicates). Reload **October 2025** from a clean source (`SELECT DISTINCT * FROM orders`): first DELETE the slice, then INSERT it again. Run both cells twice: the result must not change. Show `oct_rows` and `oct_orders` for October 2025.

*Expected result: 1 row · columns: `oct_rows`, `oct_orders`*

**How it works:** Delete the whole October slice, then insert the clean October rows. Running both again gives the same rows (idempotent).

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_orders AS
SELECT * FROM shield.orders

OK (create table as select)

In [ ]:
%%bigquery
DELETE FROM shield.prac_orders
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

OK, 5 row(s) affected

In [ ]:
%%bigquery
INSERT INTO shield.prac_orders (order_id, customer_id, order_date, amount, status)
SELECT DISTINCT order_id, customer_id, order_date, amount, status
FROM shield.orders
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

OK, 4 row(s) affected

In [ ]:
%%bigquery
SELECT COUNT(*) AS oct_rows, COUNT(DISTINCT order_id) AS oct_orders
FROM shield.prac_orders
WHERE order_date >= DATE '2025-10-01' AND order_date < DATE '2025-11-01'

,oct_rows,oct_orders
0,4,4


**Why / traps:** **Delete-then-insert by slice** (date, load id, ...) makes a job **idempotent**: a rerun replaces the slice instead of adding it twice. A plain INSERT on rerun = duplicates.

## Q6. Incremental load with a watermark  ·  🟡 medium

`prac_orders_target` holds clean orders up to 2025-06-30. Load only orders **newer than the latest date already in the target** (the watermark). Then show `num_rows` and `max_date` of the target.

*Expected result: 1 row · columns: `num_rows`, `max_date`*

**How it works:** The watermark = `MAX(order_date)` already in the target. Insert only source rows newer than it.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_orders_target AS
SELECT DISTINCT * FROM shield.orders WHERE order_date <= DATE '2025-06-30'

OK (create table as select)

In [ ]:
%%bigquery
INSERT INTO shield.prac_orders_target (order_id, customer_id, order_date, amount, status)
SELECT DISTINCT order_id, customer_id, order_date, amount, status
FROM shield.orders
WHERE order_date > (SELECT MAX(order_date) FROM shield.prac_orders_target)

OK, 30 row(s) affected

In [ ]:
%%bigquery
SELECT COUNT(*) AS num_rows, MAX(order_date) AS max_date FROM shield.prac_orders_target

,num_rows,max_date
0,104,2025-12-28


**Why / traps:** Weak spot: a row that arrives **late** with an old date is never loaded. Real pipelines use a load timestamp / load id as the watermark, or reload the last N days (idempotent).

## Q7. SCD Type 2: keep the history  ·  🔴 hard

`prac_dim_customer` has one current row per customer (`valid_to = 9999-12-31`, `is_current = 'Y'`). `prac_city_changes` says: customer 1 → Denver, customer 3 → Phoenix (no change!), customer 13 → Austin (old city is NULL). Changes start on **2025-01-15**.

1. **Close** the current row of customers whose city really changed: `valid_to = 2025-01-14`, `is_current = 'N'`.
2. **Insert** a new current row for them from 2025-01-15.
3. Show the rows of customers 1, 3, 13.

*Expected result: 5 rows · columns: `customer_id`, `city`, `valid_from`, `valid_to`, `is_current`*

**How it works:** Step 1 closes the current row of customers whose city really changed (NULL-safe compare). Step 2 inserts a new current row for every change that now has no current row.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_dim_customer AS
SELECT customer_id, city, signup_date AS valid_from,
       DATE '9999-12-31' AS valid_to, 'Y' AS is_current
FROM shield.customers

OK (create table as select)

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_city_changes AS
SELECT 1 AS customer_id, 'Denver' AS city UNION ALL
SELECT 3, 'Phoenix' UNION ALL      -- same city as now: NOT a change
SELECT 13, 'Austin'                -- old city is NULL

OK (create table as select)

In [ ]:
%%bigquery
UPDATE shield.prac_dim_customer d
SET valid_to = DATE '2025-01-14',
    is_current = 'N'
WHERE d.is_current = 'Y'
  AND EXISTS (SELECT 1
              FROM shield.prac_city_changes s
              WHERE s.customer_id = d.customer_id
                AND COALESCE(s.city, '~') <> COALESCE(d.city, '~'))

OK, 2 row(s) affected

In [ ]:
%%bigquery
INSERT INTO shield.prac_dim_customer (customer_id, city, valid_from, valid_to, is_current)
SELECT s.customer_id, s.city, DATE '2025-01-15', DATE '9999-12-31', 'Y'
FROM shield.prac_city_changes s
WHERE NOT EXISTS (SELECT 1
                  FROM shield.prac_dim_customer d
                  WHERE d.customer_id = s.customer_id
                    AND d.is_current = 'Y')

OK, 2 row(s) affected

In [ ]:
%%bigquery
SELECT customer_id, city, valid_from, valid_to, is_current
FROM shield.prac_dim_customer
WHERE customer_id IN (1, 3, 13)
ORDER BY customer_id, valid_from

,customer_id,city,valid_from,valid_to,is_current
0,1,Chicago,2023-01-26,2025-01-14,N
1,1,Denver,2025-01-15,9999-12-31,Y
2,3,Phoenix,2023-05-23,9999-12-31,Y
3,13,None,2023-10-12,2025-01-14,N
4,13,Austin,2025-01-15,9999-12-31,Y


**Why / traps:** NULL trap: `s.city <> d.city` is **unknown** when the old city is NULL, so customer 13 would be missed. COALESCE both sides (or a null-safe compare). Customer 3 must **not** get a new row. Order matters: close first, then insert the rows that have no current version.

## Q8. ALTER TABLE: add a flag column  ·  🟡 medium

Add a column `is_active` (1 letter) to `prac_customers`. Set it to `Y` for customers with at least one order, `N` for the rest. Show `is_active` and `num_customers`.

*Expected result: 2 rows · columns: `is_active`, `num_customers`*

**How it works:** `ALTER TABLE ... ADD` creates the empty column; one UPDATE fills it with CASE + EXISTS; GROUP BY checks the result.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE shield.prac_customers AS
SELECT * FROM shield.customers

OK (create table as select)

In [ ]:
%%bigquery
ALTER TABLE shield.prac_customers ADD COLUMN is_active STRING

OK (alter table)

In [ ]:
%%bigquery
UPDATE shield.prac_customers c
SET is_active = CASE WHEN EXISTS (SELECT 1 FROM shield.orders o WHERE o.customer_id = c.customer_id)
                     THEN 'Y' ELSE 'N' END
WHERE TRUE

OK, 40 row(s) affected

In [ ]:
%%bigquery
SELECT is_active, COUNT(*) AS num_customers
FROM shield.prac_customers
GROUP BY is_active
ORDER BY is_active

,is_active,num_customers
0,N,8
1,Y,32


**Why / traps:** This is a **soft delete** style flag: keep the row, mark it inactive.